# Concert Program Pattern Analysis

Explores patterns across the structured concert-program data extracted in `RareBooks_DataExtraction_rev.ipynb`: conductors, composers, and performers across ensembles, and how they change over time.

- **Source:** `Structured Data/concert_program_items.json` — the flat item export, one row per extracted venue/date/organization/patron/work/performer, each stamped with `record_type`, `page_number`, `source_text`, `review_flags`, and the manifest's `manifest_organization`/`manifest_date`.
- Counts below are **per source file**, not per individual concert. A season brochure documenting many concerts in one document (e.g. the Musashino Academia Musicae programs) contributes only one count per file for anyone appearing throughout it — a conductor who led a dozen of that season's concerts still counts as 1 — since the extracted data doesn't link a specific date/venue to a specific set of works/performers as one atomic concert. This is accurate for single-concert programs (e.g. the Melbourne Liedertafel section below) but undercounts multi-concert season brochures.

All charts use Plotly Express.


## 0. Setup

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import plotly.express as px

data_dir = Path("Structured Data")


## 1. Load the extracted items and reshape them for analysis

`concert_program_items.json` is already flat: one row per extracted `venue`/`date`/`organization`/`patron`/`work`/`performer`, tagged with `record_type` and stamped with the manifest's `manifest_organization`/`manifest_date`. We load it straight into `items_df` and derive a numeric `year` column from `manifest_date` (handling season-range values like `"1967-68"`).


In [2]:
with open(data_dir / "concert_program_items.json", encoding="utf-8") as f:
    items = json.load(f)

items_df = pd.DataFrame(items)
print(f"Loaded {len(items_df)} items across {items_df['filename'].nunique()} files")

YEAR_RE = re.compile(r"(1[5-9]\d{2}|20\d{2})")

def extract_year(date_text):
    """Pull the first plausible 4-digit year out of a free-form date string (handles season
    ranges like '1967-68' and values like 'Unknown')."""
    if not date_text:
        return None
    m = YEAR_RE.search(str(date_text))
    return int(m.group(1)) if m else None

items_df["year"] = items_df["manifest_date"].apply(extract_year)
items_df.head()


Loaded 11379 items across 87 files


,record_type,filename,source_pdf,txt_source_file,manifest_contents,manifest_date,manifest_organization,page_number,source_text,review_flags,place,date_text,name,role,composer,title,movement_or_selection,part_or_instrument,associated_work,year
0,venue,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,"Town Hall, Melbourne",[],"Town Hall, Melbourne",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1889.0
1,venue,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,20,"TOWN HALL,\nMELBOURNE",[],"TOWN HALL, MELBOURNE",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1889.0
2,date,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,"Wednesday Evening, Sixth November, 1889",[],NaN,"Wednesday Evening, Sixth November, 1889",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1889.0
3,date,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,20,"WEDNESDAY EVENING\n6TH NOVEMBER, 1889.",[],NaN,"WEDNESDAY EVENING 6TH NOVEMBER, 1889.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,1889.0
4,organization,UDC20260028-13.pdf,UDC20260028-13.pdf,extracted_texts/OLMOCR/UDC20260028-13.txt,Concert Programs,1889,Melbourne Liedertafel,3,Metropolitan Liedertafel,[],NaN,NaN,Metropolitan Liedertafel,presenting organization,NaN,NaN,NaN,NaN,NaN,1889.0


In [3]:
HONORIFIC_RE = re.compile(
    r"^(Dr|Mr|Mrs|Ms|Miss|Herr|Frau|Fräulein|Madame|Mme|Prof|Professor)\.?\s+",
    re.IGNORECASE,
)

def normalize_name(name):
    """Strip a leading honorific so 'Dr. Hans Hörner' and 'Hans Hörner' count as one person."""
    if not isinstance(name, str):
        return name
    return HONORIFIC_RE.sub("", name).strip()


## 2. Who were all the conductors of the Musashino Academia Musicae?

Matches performer records whose `part_or_instrument` contains "conduct" (`conductor`, `conductors`, `assistant conductor`, `honorary life conductor`, ...), so guest and assistant conductors are included alongside each concert's principal conductor. Names are grouped by `person_norm` (honorifics stripped), so "Dr. Hans Hörner" and "Hans Hörner" count as the same person; the table still shows every raw spelling encountered.


In [4]:
mus_performers = items_df[
    (items_df["record_type"] == "performer")
    & (items_df["manifest_organization"] == "Musashino Academia Musicae")
].copy()
mus_performers["person_norm"] = mus_performers["name"].apply(normalize_name)

conductors = mus_performers[
    mus_performers["part_or_instrument"].str.contains("conduct", case=False, na=False)
]

conductor_counts = (
    conductors.groupby("person_norm")
    .agg(
        concerts_conducted=("filename", "nunique"),
        raw_spellings=("name", lambda s: sorted(set(s))),
        roles=("part_or_instrument", lambda s: sorted(set(s))),
    )
    .sort_values("concerts_conducted", ascending=False)
    .reset_index()
)
print(f"{len(conductor_counts)} distinct conductors")
conductor_counts


26 distinct conductors


,person_norm,concerts_conducted,raw_spellings,roles
0,Tetsuya Sakuma,5,[Tetsuya Sakuma],"[Conductor, conductor]"
1,Antonin Kühnel,5,[Antonin Kühnel],"[Conductor, conductor]"
2,Tsuyoshi Sasakura,4,[Tsuyoshi Sasakura],"[Conductor, conductor]"
3,Minoru Yamada,4,[Minoru Yamada],"[Conductor, conductor]"
4,Asao Hasegawa,3,[Asao Hasegawa],"[Conductor, conductor]"
5,Tamotsu Maeda,3,[Tamotsu Maeda],[conductor]
6,Sergio Sossi,3,[Sergio Sossi],[conductor]
7,Hachiro Nanasawa,2,[Hachiro Nanasawa],[conductor]
8,Kosuke Hagiwara,2,[Kosuke Hagiwara],"[Conductor, conductor]"
9,MASAJI KATO,2,[MASAJI KATO],[conductor]


In [5]:
fig = px.bar(
    conductor_counts,
    x="concerts_conducted",
    y="person_norm",
    orientation="h",
    title="Conductors of the Musashino Academia Musicae",
    labels={"concerts_conducted": "Concerts conducted", "person_norm": "Conductor"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
fig.show()


## 3. Which composers did the Musashino Academia Musicae play most frequently?

Counts one credit per `work` record (a concert program with three pieces by three different composers contributes one count to each). `composer_norm` drops trailing annotations like ", arr. Plumpton" or ", orchestrated by ..." from the `composer` field so a work's *composer* is counted separately from whoever arranged or orchestrated it.


In [6]:
mus_works = items_df[
    (items_df["record_type"] == "work")
    & (items_df["manifest_organization"] == "Musashino Academia Musicae")
].copy()
mus_works["composer_norm"] = mus_works["composer"].fillna("").str.split(",").str[0].str.strip()
mus_works = mus_works[mus_works["composer_norm"] != ""]

composer_counts = (
    mus_works.groupby("composer_norm")
    .size()
    .reset_index(name="times_performed")
    .sort_values("times_performed", ascending=False)
)
top_composers = composer_counts.head(20)
top_composers


,composer_norm,times_performed
258,J. Brahms,102
422,R. Schumann,87
485,W. A. Mozart,85
280,J. S. Bach,73
151,F. Schubert,72
139,F. Chopin,68
329,L. v. Beethoven,55
75,C. Debussy,55
233,H. Wolf,46
199,G. Puccini,35


In [7]:
fig = px.bar(
    top_composers,
    x="times_performed",
    y="composer_norm",
    orientation="h",
    title="Most-performed composers — Musashino Academia Musicae (top 20)",
    labels={"times_performed": "Works performed", "composer_norm": "Composer"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=650)
fig.show()


## 4. Which pianists performed with which ensembles?

Matches performer records whose `part_or_instrument` contains "pian" (`piano`, `pianist`, `piano soloist`, `honorary pianist`, ...). "Ensemble" here is each program's `manifest_organization`.


In [8]:
pianists = items_df[
    (items_df["record_type"] == "performer")
    & items_df["part_or_instrument"].str.contains("pian", case=False, na=False)
].copy()
pianists["person_norm"] = pianists["name"].apply(normalize_name)

pianist_ensemble = (
    pianists.groupby(["manifest_organization", "person_norm"])
    .agg(appearances=("filename", "nunique"))
    .reset_index()
    .rename(columns={"manifest_organization": "presenter_sponsor"})
    .sort_values(["presenter_sponsor", "appearances"], ascending=[True, False])
)
print(f"{len(pianist_ensemble)} distinct pianist/ensemble pairs")
pianist_ensemble


165 distinct pianist/ensemble pairs


,presenter_sponsor,person_norm,appearances
0,Hendersen Piano Lectures Glasgow,A. M. HENDERSON,1
1,London Queens Hall Piano,SCHNABEL,1
17,Melbourne Liedertafel,G. B. FENTUM,8
19,Melbourne Liedertafel,G. B. Fentum,7
24,Melbourne Liedertafel,H. S. ELVINS.,4
...,...,...,...
160,Royal Albert Hall,FRANK ST. LEGER,1
161,Royal Albert Hall,Percy Kahn,1
163,University of Michigan,MABEL ROSS RHEAD,2
162,University of Michigan,JOSEF HOFMANN,1


In [9]:
fig = px.treemap(
    pianist_ensemble,
    path=[px.Constant("All ensembles"), "presenter_sponsor", "person_norm"],
    values="appearances",
    title="Pianists by ensemble (box size = concerts credited)",
)
fig.update_traces(root_color="lightgrey")
fig.update_layout(height=650)
fig.show()


In [10]:
# Musashino Academia Musicae dominates the dataset -- zoom in on its pianists specifically
mus_pianists = (
    pianist_ensemble[pianist_ensemble["presenter_sponsor"] == "Musashino Academia Musicae"]
    .head(20)
)

fig = px.bar(
    mus_pianists,
    x="appearances",
    y="person_norm",
    orientation="h",
    title="Pianists — Musashino Academia Musicae (top 20)",
    labels={"appearances": "Concerts", "person_norm": "Pianist"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
fig.show()


## 5. Melbourne Liedertafel violinists over time

Same `items_df` as above, filtered to `record_type == "performer"` with `manifest_organization == "Melbourne Liedertafel"` and a `part_or_instrument` mentioning "violin".

Current data covers two programs / two years (1893 and 1899). As more Melbourne Liedertafel programs are run through `RareBooks_DataExtraction_rev.ipynb`, this chart will pick them up automatically the next time `concert_program_items.json` is regenerated (Section 5 of that notebook).


In [11]:
violinists = items_df[
    (items_df["record_type"] == "performer")
    & (items_df["manifest_organization"] == "Melbourne Liedertafel")
    & items_df["part_or_instrument"].str.contains("violin", case=False, na=False)
].sort_values("year")

print(f"{len(violinists)} violinist records across {violinists['year'].nunique()} year(s)")
violinists[["year", "name", "part_or_instrument", "filename", "page_number"]]


359 violinist records across 9 year(s)


,year,name,part_or_instrument,filename,page_number
3410,1892.0,A. Bowden,First Violin,UDC20260028-25.pdf,7
3423,1892.0,Clara Ellis,Second Violin,UDC20260028-25.pdf,7
3422,1892.0,A. Wood,Second Violin,UDC20260028-25.pdf,7
3421,1892.0,R. Payne,Second Violin,UDC20260028-25.pdf,7
3420,1892.0,A. E. Garner,Second Violin,UDC20260028-25.pdf,7
...,...,...,...,...,...
6602,1910.0,""" Noske",violin,UDC20260028-42.pdf,7
6603,1910.0,""" Obbinson",violin,UDC20260028-42.pdf,7
6604,1910.0,""" Puttmann",violin,UDC20260028-42.pdf,7
6592,1910.0,Miss Chapman,violin,UDC20260028-42.pdf,7


In [12]:
fig = px.scatter(
    violinists,
    x="year",
    y="name",
    color="part_or_instrument",
    hover_data=["filename", "page_number", "source_text"],
    title="Melbourne Liedertafel violinists over time",
    labels={"year": "Year", "name": "Performer", "part_or_instrument": "Part"},
)
fig.update_layout(height=750, xaxis=dict(dtick=1))
fig.show()


## Appendix: all performer parts/roles seen

Reference for extending this analysis to new roles/questions as more programs are processed. `part_or_instrument` is the normalized field from the source JSON; `sample_ensemble` is just one organization that used it, to help spot which corpus a role tends to come from.


In [13]:
performers_df = items_df[items_df["record_type"] == "performer"].copy()
performers_df["part_or_instrument"] = performers_df["part_or_instrument"].fillna("(none)")

role_counts = (
    performers_df.groupby("part_or_instrument")
    .agg(occurrences=("filename", "size"), sample_ensemble=("manifest_organization", "first"))
    .sort_values("occurrences", ascending=False)
)
role_counts


,occurrences,sample_ensemble
part_or_instrument,,
First Bass,684,Melbourne Liedertafel
Second Bass,622,Melbourne Liedertafel
bass,528,Melbourne Liedertafel
tenor,514,Melbourne Liedertafel
First Tenor,467,Melbourne Liedertafel
...,...,...
Side Drum and Triangle,1,Melbourne Liedertafel
Side Drum,1,Melbourne Liedertafel
Rodolfo,1,Royal Albert Hall
